In [12]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import polars as pl
from PIL import Image

In [2]:
# load cpi embeddings
parquet_path = "/home/jko/ssl-cpi-analysis/data/SSL-Model-v3/cpi3m_campaign_cls_head_features_compressed.parquet"
df = pl.read_parquet(parquet_path)
unique_campaigns = df["campaign"].unique()
print("Unique campaigns:", unique_campaigns.to_list())

Unique campaigns: ['ICE_L', 'ATTREX', 'ISDAC', 'MPACE', 'CRYSTAL_FACE_UND', 'MACPEX', 'CRYSTAL_FACE_NASA', 'MIDCIX', 'ARM', 'IPHEX', 'AIRS_II', 'MC3E']


In [3]:
counts = (
    df
    .with_columns(pl.col("campaign").str.replace_all("-", "_"))
    .group_by("campaign")
    .len()
    .sort("campaign")
)
print(counts.to_pandas())
print(counts.shape)

             campaign      len
0             AIRS_II    92201
1                 ARM   295703
2              ATTREX   129128
3   CRYSTAL_FACE_NASA    78152
4    CRYSTAL_FACE_UND  1617826
5               ICE_L    46236
6               IPHEX    40692
7               ISDAC   505812
8              MACPEX    80240
9                MC3E   187558
10             MIDCIX    90761
11              MPACE    36042
(12, 2)


In [4]:
df.head()

campaign_file_id,cls_features,campaign,filename,head_features
str,list[f32],str,str,list[f32]
"""AIRS_II/1114-115708_753_18.png""","[-1.573681, -1.34678, … 1.054788]","""AIRS_II""","""1114-115708_753_18.png""","[-0.174372, 0.220068, … 0.195601]"
"""AIRS_II/1114-115708_753_25.png""","[-0.664088, -0.620979, … 1.610363]","""AIRS_II""","""1114-115708_753_25.png""","[0.150885, 0.086997, … 0.107423]"
"""AIRS_II/1114-115708_753_35.png""","[1.181937, -2.416504, … 0.973358]","""AIRS_II""","""1114-115708_753_35.png""","[0.008603, 0.284243, … 0.01234]"
"""AIRS_II/1114-115708_753_4.png""","[1.407759, -2.779922, … 0.635076]","""AIRS_II""","""1114-115708_753_4.png""","[-0.098592, 0.049973, … 0.028704]"
"""AIRS_II/1114-115745_814_121.pn…","[0.783167, -2.249387, … -0.949386]","""AIRS_II""","""1114-115745_814_121.png""","[-0.042272, -0.122378, … -0.023314]"


In [16]:
# Parse date and time from the filename

# Create unified datetime column handling both filename formats
df = df.with_columns(

    pl.when(
        pl.col("filename").str.contains(r"^\d{4}-")
    )

    # AIRS_II legacy format:
    # 1114-155011_394_15.png
    # -> 2003_1114_155011
    .then(

        (
            pl.lit("2003_")
            + pl.col("filename")
                .str.extract(r"^(\d{4}-\d{6})")
                .str.replace("-", "_")
        )

        .str.strptime(
            pl.Datetime,
            format="%Y_%m%d_%H%M%S",
            strict=False,
        )

    )

    # Standard format:
    # 2004_1017_212512_578_43.png
    .otherwise(

        pl.col("filename")
        .str.extract(r"(\d{4}_\d{4}_\d{6})")
        .str.strptime(
            pl.Datetime,
            format="%Y_%m%d_%H%M%S",
            strict=False,
        )

    )

    .alias("datetime")

)

# Optional check
print(
    df.select(["filename", "datetime"])
    .head(20)
)

shape: (20, 2)
┌─────────────────────────┬─────────────────────┐
│ filename                ┆ datetime            │
│ ---                     ┆ ---                 │
│ str                     ┆ datetime[μs]        │
╞═════════════════════════╪═════════════════════╡
│ 1114-115708_753_18.png  ┆ 2003-11-14 11:57:08 │
│ 1114-115708_753_25.png  ┆ 2003-11-14 11:57:08 │
│ 1114-115708_753_35.png  ┆ 2003-11-14 11:57:08 │
│ 1114-115708_753_4.png   ┆ 2003-11-14 11:57:08 │
│ 1114-115745_814_121.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_126.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_155.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_166.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_171.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_172.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_173.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_178.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_179.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_180.png ┆ 2003-11-14 11:57:45 │
│ 1114-115745_814_198.png ┆ 2003-11

In [17]:
# return rows where filename starts with "2004_0931_010828"
bad_rows = df.filter(
    pl.col("filename").str.starts_with("2004_0931_010828")
)

print(bad_rows)

shape: (1, 6)
┌────────────────────┬──────────────┬──────────┬────────────────────┬───────────────┬──────────────┐
│ campaign_file_id   ┆ cls_features ┆ campaign ┆ filename           ┆ head_features ┆ datetime     │
│ ---                ┆ ---          ┆ ---      ┆ ---                ┆ ---           ┆ ---          │
│ str                ┆ list[f32]    ┆ str      ┆ str                ┆ list[f32]     ┆ datetime[μs] │
╞════════════════════╪══════════════╪══════════╪════════════════════╪═══════════════╪══════════════╡
│ MPACE/2004_0931_01 ┆ [-0.744011,  ┆ MPACE    ┆ 2004_0931_010828_1 ┆ [0.019731,    ┆ null         │
│ 0828_132_10.…      ┆ 1.889707, …  ┆          ┆ 32_10.png          ┆ 0.013027, …   ┆              │
│                    ┆ 1.9517…      ┆          ┆                    ┆ 0.03092…      ┆              │
└────────────────────┴──────────────┴──────────┴────────────────────┴───────────────┴──────────────┘


There is a datetime bug since 09/31/2004 is not a real date. Investigate what is going on here...

In [18]:
# Subset MPACE campaign first
mpace_df = df.filter(
    pl.col("campaign") == "MPACE"
)

# Convert filename timestamp -> datetime
mpace_df = mpace_df.with_columns(
    pl.col("filename")
    .str.extract(r"(\d{4}_\d{4}_\d{6})")
    .str.strptime(
        pl.Datetime,
        format="%Y_%m%d_%H%M%S",
        strict=False
    )
    .alias("datetime")
)

# Get date range
min_date = mpace_df["datetime"].min()
max_date = mpace_df["datetime"].max()

print(f"Min date: {min_date}")
print(f"Max date: {max_date}")

Min date: 2004-09-30 01:01:20
Max date: 2004-10-22 00:51:01


In [19]:
# Subset MPACE and parse datetime
mpace_df = (
    df
    .filter(pl.col("campaign") == "MPACE")
    .with_columns(
        pl.col("filename")
        .str.extract(r"(\d{4}_\d{4}_\d{6})")
        .str.strptime(
            pl.Datetime,
            format="%Y_%m%d_%H%M%S",
            strict=False
        )
        .alias("datetime")
    )
)

# Count samples per calendar date
daily_counts = (
    mpace_df
    .with_columns(
        pl.col("datetime").dt.date().alias("date")
    )
    .group_by("date")
    .len()
    .sort("date")
)

pl.Config.set_tbl_rows(-1)

print(daily_counts)

shape: (15, 2)
┌────────────┬───────┐
│ date       ┆ len   │
│ ---        ┆ ---   │
│ date       ┆ u32   │
╞════════════╪═══════╡
│ null       ┆ 1     │
│ 2004-09-30 ┆ 35    │
│ 2004-10-05 ┆ 6882  │
│ 2004-10-06 ┆ 6630  │
│ 2004-10-07 ┆ 4     │
│ 2004-10-08 ┆ 603   │
│ 2004-10-09 ┆ 474   │
│ 2004-10-10 ┆ 1790  │
│ 2004-10-12 ┆ 258   │
│ 2004-10-13 ┆ 182   │
│ 2004-10-17 ┆ 13564 │
│ 2004-10-18 ┆ 1888  │
│ 2004-10-20 ┆ 533   │
│ 2004-10-21 ┆ 1707  │
│ 2004-10-22 ┆ 1491  │
└────────────┴───────┘


In [20]:
# Save daily sample size per campaign into a text file
# Assumes df already contains a correctly parsed "datetime" column

# Count samples per calendar date for each campaign
daily_counts = (
    df
    .with_columns(
        pl.col("datetime").dt.date().alias("date")
    )
    .group_by(["campaign", "date"])
    .len()
    .sort(["campaign", "date"])
)

# Write formatted output to text file
output_file = "../data/campaign_daily_counts.txt"

with open(output_file, "w") as f:

    for campaign, group in daily_counts.group_by(
        "campaign",
        maintain_order=True
    ):

        f.write(f"Campaign: {campaign}\n")

        for row in group.iter_rows(named=True):

            f.write(
                f"  {row['date']}: "
                f"{row['len']} samples\n"
            )

        f.write("\n")

print(f"Saved daily sample counts to: {output_file}")

Saved daily sample counts to: ../data/campaign_daily_counts.txt


In [21]:
# print the number of unique days in AIRS_II campaign 
airs_dates = (
    df
    .filter(pl.col("campaign") == "AIRS_II")
    .select(
        pl.col("datetime").dt.date().alias("date")
    )
    .unique()
    .sort("date")
)

print(airs_dates)

shape: (3, 1)
┌────────────┐
│ date       │
│ ---        │
│ date       │
╞════════════╡
│ null       │
│ 2003-11-14 │
│ 2003-11-19 │
└────────────┘


In [14]:
# print 10 filenames for each campaign 

# Output file
output_path = Path("../data/sample_filenames.txt")
output_path.parent.mkdir(parents=True, exist_ok=True)

# Generate 10 random filenames per campaign
samples = (
    df.group_by("campaign")
    .agg(
        pl.col("filename")
        .sample(
            n=10,
            with_replacement=False,
            shuffle=True,
        )
        .alias("sample_filenames")
    )
    .sort("campaign")
)

# Write to text file
with open(output_path, "w") as f:

    for row in samples.iter_rows(named=True):

        campaign = row["campaign"]
        filenames = row["sample_filenames"]

        f.write(f"Campaign: {campaign}\n")

        for fname in filenames:
            f.write(f"  {fname}\n")

        f.write("\n")

print(f"Saved sample filenames to: {output_path}")

Saved sample filenames to: ../data/sample_filenames.txt


In [13]:
"""
For every day with low sample counts (N<50), save a png sheet that shows all the CPI images from that day, with the campaing name and date titled at the top
CPI images saved as e.g., "/home/vanessa/hulk/cocpit/cpi_data/campaigns/MC3E/single_imgs_v1.4.0/filename.png"
"""
# Generate CPI Contact Sheets for Low-Sample Days

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
BASE_DIR = Path("/home/vanessa/hulk/cocpit/cpi_data/campaigns")
OUTPUT_DIR = Path("../data/low_sample_cpi_sheets")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
LOW_SAMPLE_THRESHOLD = 50

# ------------------------------------------------------------
# Parse datetime from filename
# ------------------------------------------------------------
df_parsed = (
    df.with_columns(
        pl.col("filename")
        .str.extract(r"(\d{4}_\d{4}_\d{6})")
        .str.strptime(
            pl.Datetime,
            format="%Y_%m%d_%H%M%S",
            strict=False,
        )
        .alias("datetime")
    )
    .with_columns(
        pl.col("datetime").dt.date().alias("date")
    )
)

# ------------------------------------------------------------
# Find low-sample campaign-days
# ------------------------------------------------------------
low_days = (
    df_parsed
    .group_by(["campaign", "date"])
    .len()
    .filter(pl.col("len") < LOW_SAMPLE_THRESHOLD)
    .filter(pl.col("date").is_not_null())
    .sort(["campaign", "date"])
)

print(low_days)

# ------------------------------------------------------------
# Generate contact sheets
# ------------------------------------------------------------
for row in low_days.iter_rows(named=True):

    campaign = row["campaign"]
    date = row["date"]
    count = row["len"]

    print(f"Processing {campaign} {date} ({count} samples)")

    subset = (
        df_parsed
        .filter(pl.col("campaign") == campaign)
        .filter(pl.col("date") == date)
        .sort("filename")
    )

    image_paths = []

    for filename in subset["filename"]:

        img_path = (
            BASE_DIR
            / campaign
            / "single_imgs_v1.4.0"
            / filename
        )

        if img_path.exists():
            image_paths.append(img_path)

    if len(image_paths) == 0:
        print(f"  No images found for {campaign} {date}")
        continue

    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------
    n_images = len(image_paths)

    ncols = min(5, n_images)
    nrows = (n_images + ncols - 1) // ncols

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(3 * ncols, 3 * nrows)
    )

    fig.suptitle(
        f"{campaign} | {date} | N={n_images}",
        fontsize=16,
        y=0.98,
    )

    # Normalize axes shape
    if nrows == 1 and ncols == 1:
        axes = [[axes]]
    elif nrows == 1:
        axes = [axes]
    elif ncols == 1:
        axes = [[ax] for ax in axes]

    flat_axes = [ax for row_axes in axes for ax in row_axes]

    # --------------------------------------------------------
    # Plot images
    # --------------------------------------------------------
    for ax, img_path in zip(flat_axes, image_paths):

        try:
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(img_path.name, fontsize=7)

        except Exception as e:
            ax.text(
                0.5,
                0.5,
                "ERROR",
                ha="center",
                va="center",
            )
            print(f"  Failed to load {img_path}: {e}")

        ax.axis("off")

    # Hide unused axes
    for ax in flat_axes[len(image_paths):]:
        ax.axis("off")

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    output_path = OUTPUT_DIR / f"{campaign}_{date}_contact_sheet.png"

    plt.savefig(output_path, dpi=200)
    plt.close(fig)

    print(f"  Saved: {output_path}")

print("Done.")

shape: (22, 3)
┌───────────────────┬────────────┬─────┐
│ campaign          ┆ date       ┆ len │
│ ---               ┆ ---        ┆ --- │
│ str               ┆ date       ┆ u32 │
╞═══════════════════╪════════════╪═════╡
│ ARM               ┆ 2000-03-06 ┆ 18  │
│ ARM               ┆ 2000-03-20 ┆ 2   │
│ CRYSTAL_FACE_NASA ┆ 2002-07-26 ┆ 24  │
│ CRYSTAL_FACE_UND  ┆ 2002-07-04 ┆ 21  │
│ CRYSTAL_FACE_UND  ┆ 2002-07-08 ┆ 11  │
│ CRYSTAL_FACE_UND  ┆ 2002-07-10 ┆ 3   │
│ CRYSTAL_FACE_UND  ┆ 2002-07-12 ┆ 1   │
│ CRYSTAL_FACE_UND  ┆ 2002-07-17 ┆ 5   │
│ CRYSTAL_FACE_UND  ┆ 2002-07-22 ┆ 14  │
│ CRYSTAL_FACE_UND  ┆ 2002-07-27 ┆ 15  │
│ ICE_L             ┆ 2007-12-04 ┆ 21  │
│ IPHEX             ┆ 2014-05-11 ┆ 12  │
│ IPHEX             ┆ 2014-05-17 ┆ 1   │
│ IPHEX             ┆ 2014-06-06 ┆ 11  │
│ IPHEX             ┆ 2014-06-09 ┆ 3   │
│ IPHEX             ┆ 2014-06-13 ┆ 7   │
│ MACPEX            ┆ 2011-04-14 ┆ 4   │
│ MC3E              ┆ 2011-05-27 ┆ 39  │
│ MIDCIX            ┆ 2004-04-23 ┆ 1   │
│